# Lab 10 (Week 2 — Lab 5): Feature Engineering

**Name:** Numair Fahad

**Role:** AI/ML Intern

**Company:** Zynvex Solutions

**GitHub:** https://github.com/numair-2003/AIML-Internship-NumairFahad

Welcome to Lab 10, the final lab of Week 2! Here you'll practice transforming raw data into better features: encoding categories, creating new features, binning, scaling, and feature selection.

**For detailed theory on each technique, see the Lab 10 PDF and Learning Resources.**

This notebook focuses on the practical, hands-on workflow.

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Compare your output with the **Expected Output** shown below each exercise where provided.

## Getting the Dataset

This lab uses the Tips dataset, built directly into Seaborn — no download or upload needed.

In [1]:
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset("tips")
print(df.shape)
df.head()

(244, 7)


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [2]:
df.dtypes

,0
total_bill,float64
tip,float64
sex,category
smoker,category
day,category
time,category
size,int64


## Example 1: Encoding Categorical Variables

In [3]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df["day_encoded"] = encoder.fit_transform(df["day"])
print(df[["day", "day_encoded"]].drop_duplicates())

     day  day_encoded
0    Sun            2
19   Sat            1
77  Thur            3
90   Fri            0


In [4]:
df_encoded = pd.get_dummies(df, columns=["day", "time"], drop_first=True)
print(df_encoded.columns.tolist())
df_encoded.head()

['total_bill', 'tip', 'sex', 'smoker', 'size', 'day_encoded', 'day_Fri', 'day_Sat', 'day_Sun', 'time_Dinner']


,total_bill,tip,sex,smoker,size,day_encoded,day_Fri,day_Sat,day_Sun,time_Dinner
0,16.99,1.01,Female,No,2,2,False,False,True,True
1,10.34,1.66,Male,No,3,2,False,False,True,True
2,21.01,3.50,Male,No,3,2,False,False,True,True
3,23.68,3.31,Male,No,2,2,False,False,True,True
4,24.59,3.61,Female,No,4,2,False,False,True,True


## Example 2: Creating a New Feature

In [5]:
df["tip_percentage"] = (df["tip"] / df["total_bill"]) * 100
df[["total_bill", "tip", "tip_percentage"]].head()

,total_bill,tip,tip_percentage
0,16.99,1.01,5.944673
1,10.34,1.66,16.054159
2,21.01,3.50,16.658734
3,23.68,3.31,13.978041
4,24.59,3.61,14.680765


## Example 3: Binning a Continuous Variable

In [6]:
df["bill_category"] = pd.cut(
    df["total_bill"],
    bins=[0, 15, 30, 100],
    labels=["Low", "Medium", "High"]
)
print(df["bill_category"].value_counts())

bill_category
Medium    132
Low        80
High       32
Name: count, dtype: int64


## Example 4: Scaling (StandardScaler vs. MinMaxScaler)

In [7]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

df["total_bill_standard"] = standard_scaler.fit_transform(df[["total_bill"]])
df["total_bill_minmax"] = minmax_scaler.fit_transform(df[["total_bill"]])

df[["total_bill", "total_bill_standard", "total_bill_minmax"]].head()

,total_bill,total_bill_standard,total_bill_minmax
0,16.99,-0.314711,0.291579
1,10.34,-1.063235,0.152283
2,21.01,0.137780,0.375786
3,23.68,0.438315,0.431713
4,24.59,0.540745,0.450775


## Example 5: Correlation for Feature Selection

In [8]:
correlation = df.select_dtypes(include=[np.number]).corr()
print(correlation["tip"].sort_values(ascending=False))

tip                    1.000000
total_bill_minmax      0.675734
total_bill             0.675734
total_bill_standard    0.675734
size                   0.489299
tip_percentage         0.342370
day_encoded           -0.011548
Name: tip, dtype: float64


## Practice Exercise 1: One-Hot Encode a Different Column

**Exercise:** Use `pd.get_dummies()` to one-hot encode the `sex` and `smoker` columns (with `drop_first=True`). Print the resulting column names.

In [9]:
### YOUR CODE HERE ###
df_encoded_2 = pd.get_dummies(df, columns=["sex", "smoker"], drop_first=True)
### END ###

print(df_encoded_2.columns.tolist())

['total_bill', 'tip', 'day', 'time', 'size', 'day_encoded', 'tip_percentage', 'bill_category', 'total_bill_standard', 'total_bill_minmax', 'sex_Female', 'smoker_No']


## Practice Exercise 2: Create a New Feature

**Exercise:** Create a new feature called `bill_per_person`, calculated as `total_bill` divided by `size` (number of people at the table). Print the first 5 rows of this new column alongside `total_bill` and `size`.

In [10]:
### YOUR CODE HERE ###
df["bill_per_person"] = df["total_bill"] / df["size"]
print(df[["total_bill", "size", "bill_per_person"]].head())
### END ###

   total_bill  size  bill_per_person
0       16.99     2         8.495000
1       10.34     3         3.446667
2       21.01     3         7.003333
3       23.68     2        11.840000
4       24.59     4         6.147500


## Practice Exercise 3: Log Transform & Comparison

**Exercise:** Apply a log transformation to the `tip` column using `np.log1p()`. Then compare the skewness of the original `tip` column to the log-transformed version using `.skew()`. Which one is less skewed?

In [11]:
### YOUR CODE HERE ###
tip_log = np.log1p(df["tip"])
original_skew = df["tip"].skew()
log_skew = tip_log.skew()
### END ###

print("Original skew:", original_skew)
print("Log-transformed skew:", log_skew)

Original skew: 1.4654510370979401
Log-transformed skew: 0.3804944462735159


### **Which tip column is less skewed?**

The log-transformed version of the `tip` feature is less skewed because it is closer to 0, compared to the original `tip` feature.

## Lab Tasks

Complete the following in this notebook, below this cell.

1. **Encoding**: Apply one-hot encoding to all categorical columns in the dataset (sex, smoker, day, time) in a single `pd.get_dummies()` call. Print the final shape and column names.
2. **Feature Creation**: Create at least 2 new derived features of your choice (e.g., bill_per_person, a weekend/weekday flag from `day`, or any other combination you find meaningful). Explain your reasoning for each in a markdown cell.
3. **Binning**: Bin the `size` column (number of people) into categories such as "Small" (1-2), "Medium" (3-4), and "Large" (5+). Print the value counts for each bin.
4. **Feature Selection**: Compute the correlation of all numeric features with `tip`. Identify the 3 features most strongly correlated with tip amount, and briefly explain in a markdown cell why you think each might be predictive.
5. **Reflection**: In a markdown cell, write 3-4 sentences on which feature engineering technique from this lab you think would have the biggest impact on model performance for this dataset, and why.

In [12]:
# Task 1
df_encoded_all = pd.get_dummies(df, columns=["sex", "smoker", "day", "time"], drop_first=True)

print("Final shape: ", df_encoded_all.shape)
print("Column names: ", df_encoded_all.columns.tolist())

Final shape:  (244, 15)
Column names:  ['total_bill', 'tip', 'size', 'day_encoded', 'tip_percentage', 'bill_category', 'total_bill_standard', 'total_bill_minmax', 'bill_per_person', 'sex_Female', 'smoker_No', 'day_Fri', 'day_Sat', 'day_Sun', 'time_Dinner']


In [13]:
# Task 2
df["bill_per_person"] = df["total_bill"] / df["size"]

df["tip_percentage"] = (df["tip"] / df["total_bill"]) * 100

df["is_weekend"] = df["day"].isin(["Sat", "Sun"]).astype(int)

print(df[["total_bill", "tip", "size", "bill_per_person", "day", "is_weekend", "tip_percentage"]].head())

   total_bill   tip  size  bill_per_person  day  is_weekend  tip_percentage
0       16.99  1.01     2         8.495000  Sun           1        5.944673
1       10.34  1.66     3         3.446667  Sun           1       16.054159
2       21.01  3.50     3         7.003333  Sun           1       16.658734
3       23.68  3.31     2        11.840000  Sun           1       13.978041
4       24.59  3.61     4         6.147500  Sun           1       14.680765


### **Justification**

1. **`bill_per_person`**: Dividing the total bill by the number of people at the dinner table (size) gives the average bill to be spent per person, which is more informative than the raw total bill alone.

2. **`is_weekend`**: Tips and spending patterns often vary on weekends compared to weekdays. Therefore, converting each day into a simple weekend/weekday flag can help the model capture this difference.

3. **`tip_percentage`**: The tip as a percentage of the bill is viewed as a better indicator of tipping behavior than the absolute, raw tip amount.

In [14]:
# Task 3
df["size_category"] = pd.cut(
    df["size"],
    bins=[0, 2, 4, 10],
    labels=["Small", "Medium", "Large"]
)

print(df["size_category"].value_counts())

size_category
Small     160
Medium     75
Large       9
Name: count, dtype: int64


In [16]:
# Task 4
correlation = df.select_dtypes(include=[np.number]).corr()
print(correlation["tip"].sort_values(ascending=False))

tip                    1.000000
total_bill_minmax      0.675734
total_bill             0.675734
total_bill_standard    0.675734
size                   0.489299
bill_per_person        0.347393
tip_percentage         0.342370
is_weekend             0.120198
day_encoded           -0.011548
Name: tip, dtype: float64


### **Three features strongly and positively correlated with the tip amount**

1. **`total_bill`** (**~0.675**): Generation of larger bills mostly lead to larger tip amounts, which is considered the strongest and most direct relationship between `total_bill` and `tip`.

2. **`size`** (**~0.489**): More people at the dinner table means a higher total bill is to be paid, which further results in a higher tip amount.

3. **`bill_per_person`** (**~0.347**): This derived feature represents the average bill spent per person and captures spending behavior more precisely. It helps explain tip variation rather than just the raw total bill.

## **Task 5**

Among the feature engineering techniques practiced in this lab, **engineering/creating new derived features** (e.g. **tip percentage** and **bill per person**) would have the **biggest impact** on **model performance** because although **raw features** including **total_bill** and **tip** contain useful information, combining them into meaningful ratios **reveals clearer patterns for the model**. Encoding categorical variables is also significant, but without strong numerical features, the model still has limited patterns to learn from, which is why creating new derived features from already available features in the provided dataset is important.

In [17]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/numair-2003/AIML-Internship-NumairFahad.git

Mounted at /content/drive
Cloning into 'AIML-Internship-NumairFahad'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 99 (delta 46), reused 47 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 1.46 MiB | 11.75 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/YourNotebook.ipynb" "/content/your-repo/"